# Relação entre fatias da imagem e círculos detectados na aorta

## Objetivo

Avalia a relação entre a quantidade de fatias do volume e os círculos detectados durante a localização da aorta.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from utils.project.notebook_env import configure_notebook_environment
    REPO_ROOT = configure_notebook_environment(chdir_to_src=False)
except Exception:
    current = Path.cwd().resolve()
    REPO_ROOT = next(
        path for path in [current, *current.parents]
        if (path / "src").exists() and (path / "output").exists()
    )

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

ANALYSIS_DIR = REPO_ROOT / "output/segmentation/analysis/aorta_circle_slices"
FIGURE_DIR = ANALYSIS_DIR / "figures"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

ANALYSIS_DIR.relative_to(REPO_ROOT)


## Configuração

Se quiser analisar um run específico, preencha `RUN_SUMMARY_PATH`. Se deixar `None`, o notebook procura automaticamente o CSV mais recente que já possui as colunas `image_slice_count` e `aorta_circle_count`. Runs antigos podem não ter essas colunas; nesse caso, rode novamente o pipeline depois da atualização que salva essas métricas.

In [ ]:
RUN_SUMMARY_PATH = None
# Exemplo:
# RUN_SUMMARY_PATH = REPO_ROOT / "output/segmentation/runs/mid_res/2026-06-XX_XX-XX-XX/numeric/ostios_test_summary.csv"

REQUIRED_COLUMNS = {"IMG_ID", "image_slice_count", "aorta_circle_count"}
OPTIONAL_COLUMNS = [
    "aorta_detected_circle_count",
    "aorta_interpolated_circle_count",
    "aorta_circle_coverage",
    "aorta_circle_first_slice",
    "aorta_circle_last_slice",
    "artery_dice",
    "ostia_detection_status",
]


def find_latest_summary_with_circle_columns(root: Path) -> Path | None:
    candidates = sorted(
        root.glob("output/segmentation/**/numeric/ostios_*_summary.csv"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    for path in candidates:
        try:
            columns = set(pd.read_csv(path, nrows=0).columns)
        except Exception:
            continue
        if REQUIRED_COLUMNS.issubset(columns):
            return path
    return None


if RUN_SUMMARY_PATH is None:
    RUN_SUMMARY_PATH = find_latest_summary_with_circle_columns(REPO_ROOT)

if RUN_SUMMARY_PATH is None:
    raise FileNotFoundError(
        "Nenhum ostios_*_summary.csv com image_slice_count e aorta_circle_count foi encontrado. "
        "Rode novamente o pipeline para gerar essas colunas."
    )

RUN_SUMMARY_PATH = Path(RUN_SUMMARY_PATH)
print(f"Resumo selecionado: {RUN_SUMMARY_PATH.relative_to(REPO_ROOT)}")


## Carregamento e métricas derivadas

In [ ]:
df_raw = pd.read_csv(RUN_SUMMARY_PATH)
missing = REQUIRED_COLUMNS - set(df_raw.columns)
if missing:
    raise ValueError(
        f"O arquivo selecionado não possui as colunas {sorted(missing)}. "
        "Rode o pipeline novamente ou selecione um run mais recente."
    )

df = df_raw.copy()
for column in ["image_slice_count", "aorta_circle_count", *OPTIONAL_COLUMNS]:
    if column in df.columns:
        converted = pd.to_numeric(df[column], errors="coerce")
        if converted.notna().any():
            df[column] = converted

if "aorta_detected_circle_count" not in df.columns:
    df["aorta_detected_circle_count"] = df["aorta_circle_count"]
if "aorta_interpolated_circle_count" not in df.columns:
    df["aorta_interpolated_circle_count"] = 0

df["aorta_circle_coverage"] = df["aorta_circle_count"] / df["image_slice_count"]
df["aorta_detected_circle_coverage"] = df["aorta_detected_circle_count"] / df["image_slice_count"]
df["aorta_missing_circle_slices"] = df["image_slice_count"] - df["aorta_circle_count"]
df["aorta_interpolated_circle_fraction"] = np.where(
    df["aorta_circle_count"] > 0,
    df["aorta_interpolated_circle_count"] / df["aorta_circle_count"],
    np.nan,
)

analysis_columns = [
    "IMG_ID",
    "image_slice_count",
    "aorta_circle_count",
    "aorta_detected_circle_count",
    "aorta_interpolated_circle_count",
    "aorta_circle_coverage",
    "aorta_detected_circle_coverage",
    "aorta_missing_circle_slices",
    "aorta_interpolated_circle_fraction",
]
for column in ["artery_dice", "ostia_detection_status", "aorta_circle_first_slice", "aorta_circle_last_slice"]:
    if column in df.columns and column not in analysis_columns:
        analysis_columns.append(column)

analysis_df = df[analysis_columns].copy()
analysis_path = ANALYSIS_DIR / "aorta_circle_slice_metrics.csv"
analysis_df.to_csv(analysis_path, index=False)
print(f"CSV salvo em: {analysis_path.relative_to(REPO_ROOT)}")
display(analysis_df.head())


## Análise

As subseções abaixo apresentam as métricas, tabelas ou visualizações do objetivo definido.

## Estatísticas gerais

In [ ]:
numeric_columns = [
    "image_slice_count",
    "aorta_circle_count",
    "aorta_detected_circle_count",
    "aorta_interpolated_circle_count",
    "aorta_circle_coverage",
    "aorta_detected_circle_coverage",
    "aorta_missing_circle_slices",
    "aorta_interpolated_circle_fraction",
]
if "artery_dice" in analysis_df.columns:
    numeric_columns.append("artery_dice")

stats_df = analysis_df[numeric_columns].describe().T
stats_path = ANALYSIS_DIR / "aorta_circle_slice_stats.csv"
stats_df.to_csv(stats_path)
print(f"CSV salvo em: {stats_path.relative_to(REPO_ROOT)}")
display(stats_df.round(4))


## Correlação

A correlação principal é entre `image_slice_count` e `aorta_circle_count`. Também vale olhar a cobertura (`aorta_circle_coverage`), porque uma imagem com mais fatias naturalmente pode ter mais círculos mesmo que a proporção localizada seja parecida.

In [ ]:
corr_columns = [
    "image_slice_count",
    "aorta_circle_count",
    "aorta_detected_circle_count",
    "aorta_circle_coverage",
    "aorta_detected_circle_coverage",
    "aorta_missing_circle_slices",
]
if "artery_dice" in analysis_df.columns:
    corr_columns.append("artery_dice")

pearson_corr = analysis_df[corr_columns].corr(method="pearson")
spearman_corr = analysis_df[corr_columns].corr(method="spearman")
pearson_corr.to_csv(ANALYSIS_DIR / "aorta_circle_slice_pearson_corr.csv")
spearman_corr.to_csv(ANALYSIS_DIR / "aorta_circle_slice_spearman_corr.csv")

print("Pearson")
display(pearson_corr.round(4))
print("Spearman")
display(spearman_corr.round(4))


## Dispersão: fatias da imagem vs círculos da aorta

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
scatter_kwargs = {"s": 28, "alpha": 0.75, "edgecolor": "white", "linewidth": 0.4}
if "artery_dice" in analysis_df.columns:
    points = ax.scatter(
        analysis_df["image_slice_count"],
        analysis_df["aorta_circle_count"],
        c=analysis_df["artery_dice"],
        cmap="viridis",
        **scatter_kwargs,
    )
    cbar = fig.colorbar(points, ax=ax)
    cbar.set_label("Dice da artéria")
else:
    ax.scatter(
        analysis_df["image_slice_count"],
        analysis_df["aorta_circle_count"],
        color="#357ABD",
        **scatter_kwargs,
    )

ax.set_xlabel("Número de fatias da imagem")
ax.set_ylabel("Número de círculos da aorta")
ax.grid(True, alpha=0.25)
fig.tight_layout()
figure_path = FIGURE_DIR / "image_slices_vs_aorta_circles.png"
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
print(f"Figura salva em: {figure_path.relative_to(REPO_ROOT)}")
plt.show()


## Cobertura relativa da localização da aorta

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    analysis_df["image_slice_count"],
    analysis_df["aorta_circle_coverage"],
    s=28,
    alpha=0.75,
    color="#C44E52",
    edgecolor="white",
    linewidth=0.4,
)
ax.set_xlabel("Número de fatias da imagem")
ax.set_ylabel("Círculos / fatias da imagem")
ax.set_ylim(0, max(1.05, analysis_df["aorta_circle_coverage"].max() * 1.05))
ax.grid(True, alpha=0.25)
fig.tight_layout()
figure_path = FIGURE_DIR / "aorta_circle_coverage.png"
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
print(f"Figura salva em: {figure_path.relative_to(REPO_ROOT)}")
plt.show()


## Casos com menor quantidade relativa de círculos

In [ ]:
low_coverage_df = analysis_df.sort_values(
    ["aorta_circle_coverage", "aorta_circle_count"],
    ascending=[True, True],
).head(30)
low_coverage_path = ANALYSIS_DIR / "lowest_aorta_circle_coverage_cases.csv"
low_coverage_df.to_csv(low_coverage_path, index=False)
print(f"CSV salvo em: {low_coverage_path.relative_to(REPO_ROOT)}")
display(low_coverage_df)


## Relação com status dos óstios

In [ ]:
if "ostia_detection_status" in analysis_df.columns:
    status_summary = (
        analysis_df.groupby("ostia_detection_status")
        .agg(
            n_images=("IMG_ID", "count"),
            mean_image_slices=("image_slice_count", "mean"),
            mean_aorta_circles=("aorta_circle_count", "mean"),
            median_aorta_circles=("aorta_circle_count", "median"),
            mean_circle_coverage=("aorta_circle_coverage", "mean"),
            mean_detected_circle_coverage=("aorta_detected_circle_coverage", "mean"),
        )
        .sort_values("mean_circle_coverage", ascending=False)
    )
    status_path = ANALYSIS_DIR / "aorta_circle_by_ostia_status.csv"
    status_summary.to_csv(status_path)
    print(f"CSV salvo em: {status_path.relative_to(REPO_ROOT)}")
    display(status_summary.round(4))
else:
    print("Coluna ostia_detection_status não encontrada neste resumo.")


## Conclusão

As métricas e gráficos permitem verificar se a cobertura de círculos acompanha o número de fatias e se coberturas baixas coincidem com falhas nos óstios.